---
# Chapter 14 — Context Is a Bottleneck

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 14: Context Is a Bottleneck |
| Central question | When candidate set exceeds budget, what survives assembly and why? |
| Main concepts | Context assembly, Bounded context, Evidence sufficiency, Reader sufficiency, ContextTrace |
| Implementation | context_frames (assembly policy) |
| Experiment | ch14-20260920T174259Z-context-assembly |
| Evidence status | Book result: core for bounded context |
| Depends on | Chapter 10 (frames), Chapter 12 (behavioural hinge), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This chapter completes the separation: **Memory is durable. Context is selected.** The notebook:

1. **Loads the frozen context assembly run** (`ch14-20260920T174259Z-context-assembly`)
2. **Starts with a candidate set larger than the budget** and runs the real assembler
3. **Inspects which items survive and why** via `ContextTrace`
4. **Compares earned assembly policy against random dropping** using committed experiment data
5. **Shows the answer/evidence divergence**: ledger evidence recall fell but behavioural score rose (substitution + conservative ledger)
6. **Keeps evidence sufficiency separate from reader sufficiency**

> **Evidence status**: Book result. Assembly preserves disagreement/licences under hard budgets and reduced cost. Answer/evidence divergence resolved as substitution plus conservative ledger. Assembly scored highest of practical conditions behaviourally at ~3/5 tokens of retrieval.

## The chapter question

> **What survives a bounded context?**

The store may contain years of history. The reader receives a bounded working set. Assembly deserves explicit treatment — not because deduplication unlocked huge efficiency (dedup saved only 18 of 1,174 tokens) — but because the composed policy did better than random dropping at preserving contradiction and derived licences under a hard budget.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(14)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 14 Concepts")

## Load the frozen context assembly run

In [ ]:
from notebooks.memory._support import load_frozen_run

run = load_frozen_run("ch14-20260920T174259Z-context-assembly")
metrics = run["metrics"]

print(f"Run ID: {run['run_id']}")
print(f"Policies: {sorted(metrics.keys())}")

## The assembly policy (from the chapter)

The real assembler runs over a candidate set larger than the budget. Key policy elements:

1. **Deduplication**: Exact text duplicates suppressed (saved 18 of 1,174 tokens in frozen run)
",2. **Frame-conditioned eligibility**: Only items relevant to current WorkFrame/ProjectFrame
",3. **Temporal validity**: Superseded items marked, not admitted for current-state questions
",4. **Evidence licence**: Derived obligations (Chapter 11) admitted with their licence chain
",5. **Budget enforcement**: Hard token limit; breach recorded, never silently exceeded
",6. **Contradiction preservation**: Items that contradict each other both admitted if both licensed
",7. **Trust gate (Chapter 17)**: Authority check before admission
",
> **Key finding**: The composed policy did better than random dropping at preserving contradiction and derived licences under a hard budget, but reader answer coverage and ledger evidence recall diverged sharply.

## ContextTrace — the audit trail

`ContextTrace` records:

- Admitted passages (with source IDs)
- Duplicate drops
- Budget drops
- Character count & estimated tokens
- Source coverage
- Eligibility decisions (why each candidate was admitted/rejected)

This is required to explain selected and rejected candidates and to localise failures in retrieval, eligibility, budget, and policy.

In [ ]:
# A candidate pool larger than the budget through the real assembler.
# Retrieval proposes; this stage disposes, and the trace says why.
from memory_baseline.context import ContextConfig, assemble
from memory_baseline.storage import ScoredChunk

texts = [
    ("adr-007", "DECIDED: New event-store work targets PostgreSQL."),
    ("session-035", "Benchmark: PostgreSQL handles 10x concurrent writes."),
    ("session-033", "Contention observed in the SQLite prototype."),
    ("session-031", "Proposal: use SQLite for simplicity."),
    ("session-040", "Proposal: introduce Redis for caching."),
    ("adr-009", "DECIDED: Do not introduce Redis."),
    ("fact-301", "Production ran SQLite until 22 July (superseded)."),
    ("fact-302", "Production runs PostgreSQL since 22 July (current)."),
]
candidates = [
    ScoredChunk(chunk_id=f"{sid}-{i}", source_id=sid, text=t,
                section=None, score=0.95 - 0.05 * i, rank=i + 1)
    for i, (sid, t) in enumerate(texts)
]

context = assemble(candidates, ContextConfig(max_chars=800, max_passages=3))

print(f"Admitted: {len(context.admitted)} passages")
print(f"Admitted chars: {context.admitted_chars}")
print(f"Estimated tokens: {context.admitted_tokens_estimate}")
print(f"Sources covered: {context.sources_covered}")
print(f"Dropped duplicates: {context.dropped_duplicates}")
print(f"Dropped over budget: {context.dropped_over_budget}")
print()
for chunk in context.admitted:
    print(f"  [rank {chunk.rank}] {chunk.source_id}: {chunk.text[:70]}...")

## The answer/evidence divergence

The chapter's central finding:

> **Ledger evidence recall fell but behavioural score rose.** The divergence resolved as **substitution plus a conservative ledger** — an unlabelled open-loop record carried the same actionable content as a missing required unit, and the behavioural run confirmed it by holding the chapter for the right reason under reduced context.
",
**Evidence sufficiency ≠ Reader sufficiency**. Cleaner context is better memory only where the behaviour agrees.

In [ ]:
# The frozen verdict at a hard 384-token budget: the composed policy
# preserves contradiction and derived licences; random dropping does not.
BUDGET = "384"
render_table(
    [{"Policy": p,
      "Required recall": metrics[p][BUDGET]["required_recall"],
      "Contradiction preserved": metrics[p][BUDGET]["contradiction_preservation"],
      "Licences preserved": metrics[p][BUDGET]["licence_preservation"],
      "Harmful admitted": metrics[p][BUDGET]["harmful_admission"],
      "Counted tokens": metrics[p][BUDGET]["counted_tokens"]}
     for p in ("A6-composed", "random-drop", "A0-raw")],
    "Assembly policies at 384 tokens (frozen ch14 metrics)")

## Comparison: Earned assembly vs random dropping

The frozen run compared the composed assembly policy against controls including random dropping. The earned policy:

- Preserved contradiction where both sides were licensed
- Preserved derived obligation chains (Chapter 11)
- Enforced temporal validity (superseded items not admitted for current-state)
- Enforced trust gate (Chapter 17)
- Recorded every drop in ContextTrace for auditability

Random dropping failed on all these dimensions.

## What this establishes

- **Memory is durable; context is selected** — fundamental architectural separation
- **Assembly is an optimisation and safety layer**, not proof that the model reasons better
- **Deduplication alone is trivial** (18/1,174 tokens) — the value is in *which* items survive
- **Evidence sufficiency and reader sufficiency are different** — substitution occurs
- **ContextTrace is core for auditable memory** — required to explain every selection/rejection
- **The behavioural run confirmed the substitution hypothesis** — the system held the chapter for the right reason under reduced context

## What this does NOT establish

- No huge efficiency win from deduplication
- Assembly does not improve mean answer correctness over strong RAG
- The divergence means we cannot use evidence recall as a proxy for behavioural utility
- Real-corpus extraction quality for assembly remains untested

## Try it yourself

Modify the budget or candidate set above and observe how the admission changes. Key dials:

- `max_chars` / `max_passages` in `ContextConfig`
- Add/remove candidates with temporal status (current vs superseded)
- Add candidates with/without derived licence chains

In [ ]:
# TRY IT YOURSELF: vary the budget and watch admission change.
print("=== Varying Context Budget ===\n")
for budget_chars in (400, 800, 1600, 3200):
    ctx = assemble(candidates,
                   ContextConfig(max_chars=budget_chars, max_passages=6))
    print(f"Budget {budget_chars:4d} chars: {len(ctx.admitted)} admitted, "
          f"{ctx.admitted_tokens_estimate} est. tokens, "
          f"sources: {ctx.sources_covered}")

## Where this leads next

Chapter 15 asks: **What should memory keep over the long term?** — consolidation, compression, forgetting as growth policies (experimental/deferred).

Chapter 16 asks: **When does remembering become learning?** — outcome adaptation, procedure extraction (experimental/deferred).

Chapter 17 asks: **Can memory be trusted?** — staged admission gate, quarantine, revocation.

Chapter 18 composes the earned architecture end-to-end.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\14-chapter.ipynb)